# HW08-09: PyTorch MLP — регуляризация и оптимизация

- Датасет: **FashionMNIST** (28×28, 10 классов; устойчивее к загрузке, чем KMNIST).
- Часть A: E1–E4; часть B: O1–O3.
- Запускайте ноутбук из каталога `homeworks/HW08-09/` (Run All).

In [1]:
# HW08-09 — FashionMNIST MLP: регуляризация и оптимизация (единый ноутбук)
import os
import json
import csv
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

# --- артефакты в homeworks/HW08-09/artifacts (cwd = каталог ноутбука или поднимаемся из корня репо) ---
import pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name != "HW08-09":
    cand = ROOT / "homeworks" / "HW08-09"
    if cand.is_dir():
        os.chdir(cand)
        ROOT = pathlib.Path.cwd().resolve()
ART = str(ROOT / "artifacts")
FIG = str(ROOT / "artifacts" / "figures")
os.makedirs(FIG, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATASET_NAME = "FashionMNIST"
DATA_ROOT = "data"
transform = transforms.Compose([transforms.ToTensor()])
train_full = torchvision.datasets.FashionMNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.FashionMNIST(root=DATA_ROOT, train=False, download=True, transform=transform)

n_val = int(0.2 * len(train_full))
n_tr = len(train_full) - n_val
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [n_tr, n_val], generator=g)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x, y = next(iter(train_loader))
print("Sanity:", "x", tuple(x.shape), "y", tuple(y.shape), "x in [", float(x.min()), ",", float(x.max()), "]")

# размер входа
INPUT_SIZE = 28 * 28
NUM_CLASSES = 10
HIDDEN_SIZES = (256, 128)


class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout_p=0.0, use_batchnorm=False):
        super().__init__()
        layers = []
        prev = INPUT_SIZE
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p and dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            prev = h
        self.net = nn.Sequential(nn.Flatten(), *layers, nn.Linear(prev, NUM_CLASSES))

    def forward(self, x):
        return self.net(x)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    tot_loss = tot = correct = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item() * xb.size(0)
        tot += xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
    return tot_loss / tot, correct / tot


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot_loss = tot = correct = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        tot_loss += loss.item() * xb.size(0)
        tot += xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
    return tot_loss / tot, correct / tot


def run_epochs(model, opt, epochs, tag=""):
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    for ep in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, criterion, opt)
        vl, va = evaluate(model, val_loader, criterion)
        for k, v in zip(hist.keys(), (tl, ta, vl, va)):
            hist[k].append(v)
        if (ep + 1) % max(1, epochs // 5) == 0 or ep == 0:
            print(f"  {tag} ep{ep+1}/{epochs}  val_acc={va:.4f} val_loss={vl:.4f}")
    best_acc = max(hist["val_acc"])
    best_loss = min(hist["val_loss"])
    return best_acc, best_loss, hist


def model_summary_str(dropout_p, use_bn):
    s = f"MLP {HIDDEN_SIZES[0]}-{HIDDEN_SIZES[1]} ReLU"
    if use_bn:
        s += " + BatchNorm1d (after Linear)"
    if dropout_p and dropout_p > 0:
        s += f" + Dropout({dropout_p})"
    return s


criterion = nn.CrossEntropyLoss()
MAX_EPOCHS_A = 15
LR_BASE = 1e-3

print("E1 base")
m1 = MLP(HIDDEN_SIZES, 0.0, False).to(device)
o1 = torch.optim.Adam(m1.parameters(), lr=LR_BASE)
e1_acc, e1_loss, h1 = run_epochs(m1, o1, MAX_EPOCHS_A, "E1")

print("E2 dropout")
m2 = MLP(HIDDEN_SIZES, 0.3, False).to(device)
o2 = torch.optim.Adam(m2.parameters(), lr=LR_BASE)
e2_acc, e2_loss, h2 = run_epochs(m2, o2, MAX_EPOCHS_A, "E2")

print("E3 batchnorm")
m3 = MLP(HIDDEN_SIZES, 0.0, True).to(device)
o3 = torch.optim.Adam(m3.parameters(), lr=LR_BASE)
e3_acc, e3_loss, h3 = run_epochs(m3, o3, MAX_EPOCHS_A, "E3")

best_ab = "E2" if e2_acc >= e3_acc else "E3"
USE_DROPOUT_E4 = best_ab == "E2"
USE_BN_E4 = best_ab == "E3"
DROPOUT_P_E4 = 0.3 if USE_DROPOUT_E4 else 0.0
print("Best E2 vs E3:", best_ab, "val_acc", max(e2_acc, e3_acc))

PATIENCE = 4
print("E4 early stopping on architecture of", best_ab)
m4 = MLP(HIDDEN_SIZES, DROPOUT_P_E4, USE_BN_E4).to(device)
o4 = torch.optim.Adam(m4.parameters(), lr=LR_BASE)
best_val_acc = 0.0
best_val_loss = float("inf")
best_epoch = 0
pat = 0
h4 = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
for ep in range(MAX_EPOCHS_A):
    tl, ta = train_one_epoch(m4, train_loader, criterion, o4)
    vl, va = evaluate(m4, val_loader, criterion)
    for k, v in zip(h4.keys(), (tl, ta, vl, va)):
        h4[k].append(v)
    if va > best_val_acc:
        best_val_acc = va
        best_val_loss = vl
        best_epoch = ep + 1
        pat = 0
        torch.save(m4.state_dict(), os.path.join(ART, "best_model.pt"))
    else:
        pat += 1
    if pat >= PATIENCE:
        print(f"EarlyStopping at epoch {ep+1}, best val_acc={best_val_acc:.4f} @ {best_epoch}")
        break
epochs_e4 = len(h4["val_acc"])

EPOCHS_LR = 7
EPOCHS_O3 = 12
print("O1 large LR")
mo1 = MLP(HIDDEN_SIZES, DROPOUT_P_E4, USE_BN_E4).to(device)
oo1 = torch.optim.Adam(mo1.parameters(), lr=0.1)
o1_acc, o1_loss, ho1 = run_epochs(mo1, oo1, EPOCHS_LR, "O1")

print("O2 tiny LR")
mo2 = MLP(HIDDEN_SIZES, DROPOUT_P_E4, USE_BN_E4).to(device)
oo2 = torch.optim.Adam(mo2.parameters(), lr=1e-5)
o2_acc, o2_loss, ho2 = run_epochs(mo2, oo2, EPOCHS_LR, "O2")

print("O3 SGD+momentum+wd")
mo3 = MLP(HIDDEN_SIZES, DROPOUT_P_E4, USE_BN_E4).to(device)
oo3 = torch.optim.SGD(mo3.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
o3_acc, o3_loss, ho3 = run_epochs(mo3, oo3, EPOCHS_O3, "O3")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(1, len(ho1["val_loss"]) + 1), ho1["val_loss"], label="val_loss")
axes[0].plot(range(1, len(ho1["train_loss"]) + 1), ho1["train_loss"], label="train_loss", alpha=0.7)
axes[0].set_title("O1: Adam lr=0.1")
axes[0].legend()
axes[0].set_xlabel("epoch")
axes[1].plot(range(1, len(ho2["val_loss"]) + 1), ho2["val_loss"], label="val_loss")
axes[1].plot(range(1, len(ho2["train_loss"]) + 1), ho2["train_loss"], label="train_loss", alpha=0.7)
axes[1].set_title("O2: Adam lr=1e-5")
axes[1].legend()
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig(os.path.join(FIG, "curves_lr_extremes.png"), dpi=120)
plt.close()

ep = range(1, len(h4["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ep, h4["train_loss"], label="train")
axes[0].plot(ep, h4["val_loss"], label="val")
axes[0].set_title("E4 loss")
axes[0].legend()
axes[1].plot(ep, h4["train_acc"], label="train")
axes[1].plot(ep, h4["val_acc"], label="val")
axes[1].set_title("E4 accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG, "curves_best.png"), dpi=120)
plt.close()

rows = [
    {"experiment_id": "E1", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(0.0, False), "optimizer": "Adam", "lr": LR_BASE,
     "momentum": "", "weight_decay": 0, "epochs_trained": MAX_EPOCHS_A,
     "best_val_accuracy": e1_acc, "best_val_loss": e1_loss},
    {"experiment_id": "E2", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(0.3, False), "optimizer": "Adam", "lr": LR_BASE,
     "momentum": "", "weight_decay": 0, "epochs_trained": MAX_EPOCHS_A,
     "best_val_accuracy": e2_acc, "best_val_loss": e2_loss},
    {"experiment_id": "E3", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(0.0, True), "optimizer": "Adam", "lr": LR_BASE,
     "momentum": "", "weight_decay": 0, "epochs_trained": MAX_EPOCHS_A,
     "best_val_accuracy": e3_acc, "best_val_loss": e3_loss},
    {"experiment_id": "E4", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(DROPOUT_P_E4, USE_BN_E4) + "; EarlyStopping",
     "optimizer": "Adam", "lr": LR_BASE, "momentum": "", "weight_decay": 0,
     "epochs_trained": epochs_e4, "best_val_accuracy": best_val_acc, "best_val_loss": best_val_loss},
    {"experiment_id": "O1", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(DROPOUT_P_E4, USE_BN_E4), "optimizer": "Adam", "lr": 0.1,
     "momentum": "", "weight_decay": 0, "epochs_trained": EPOCHS_LR,
     "best_val_accuracy": o1_acc, "best_val_loss": o1_loss},
    {"experiment_id": "O2", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(DROPOUT_P_E4, USE_BN_E4), "optimizer": "Adam", "lr": 1e-5,
     "momentum": "", "weight_decay": 0, "epochs_trained": EPOCHS_LR,
     "best_val_accuracy": o2_acc, "best_val_loss": o2_loss},
    {"experiment_id": "O3", "dataset": DATASET_NAME, "seed": SEED,
     "model_summary": model_summary_str(DROPOUT_P_E4, USE_BN_E4), "optimizer": "SGD", "lr": 0.01,
     "momentum": 0.9, "weight_decay": 1e-4, "epochs_trained": EPOCHS_O3,
     "best_val_accuracy": o3_acc, "best_val_loss": o3_loss},
]

with open(os.path.join(ART, "runs.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

best = MLP(HIDDEN_SIZES, DROPOUT_P_E4, USE_BN_E4).to(device)
best.load_state_dict(torch.load(os.path.join(ART, "best_model.pt"), map_location=device))
test_loss, test_acc = evaluate(best, test_loader, criterion)
print("Test (once):", test_acc, test_loss)

cfg = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "hidden_sizes": list(HIDDEN_SIZES),
    "dropout_p": DROPOUT_P_E4,
    "use_batchnorm": USE_BN_E4,
    "optimizer": "Adam",
    "lr": LR_BASE,
    "batch_size": BATCH_SIZE,
    "early_stopping_patience": PATIENCE,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_acc,
    "best_val_loss": best_val_loss,
    "test_accuracy": test_acc,
    "test_loss": test_loss,
    "saved_at": datetime.now().isoformat(),
}
with open(os.path.join(ART, "best_config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)
print("Artifacts saved under", ART)


Device: cpu
Sanity: x (256, 1, 28, 28) y (256,) x in [ 0.0 , 1.0 ]
E1 base
  E1 ep1/15  val_acc=0.8310 val_loss=0.4790
  E1 ep3/15  val_acc=0.8635 val_loss=0.3848
  E1 ep6/15  val_acc=0.8680 val_loss=0.3617
  E1 ep9/15  val_acc=0.8849 val_loss=0.3247
  E1 ep12/15  val_acc=0.8878 val_loss=0.3150
  E1 ep15/15  val_acc=0.8910 val_loss=0.3084
E2 dropout
  E2 ep1/15  val_acc=0.8319 val_loss=0.4691
  E2 ep3/15  val_acc=0.8599 val_loss=0.3820
  E2 ep6/15  val_acc=0.8766 val_loss=0.3365
  E2 ep9/15  val_acc=0.8825 val_loss=0.3213
  E2 ep12/15  val_acc=0.8866 val_loss=0.3168
  E2 ep15/15  val_acc=0.8892 val_loss=0.3072
E3 batchnorm
  E3 ep1/15  val_acc=0.8658 val_loss=0.3827
  E3 ep3/15  val_acc=0.8608 val_loss=0.3849
  E3 ep6/15  val_acc=0.8638 val_loss=0.3807
  E3 ep9/15  val_acc=0.8893 val_loss=0.3233
  E3 ep12/15  val_acc=0.8669 val_loss=0.4018
  E3 ep15/15  val_acc=0.8878 val_loss=0.3753
Best E2 vs E3: E2 val_acc 0.8898333333333334
E4 early stopping on architecture of E2
O1 large LR
  O1 e